# Aliases de campo e valores default

Até agora definimos nomes, tipos e o comportamento de validação dos campos só com *type hints* e valores default.

Mas o Pydantic tem uma forma mais poderosa de anexar configuração aos campos do modelo: o objeto `Field`.

Para configurar campos além do que já vimos, criamos uma instância de `Field`, configuramos o que queremos nela e a anexamos ao campo
usando-a como valor default.

Isso significa que precisamos de outra forma para especificar `Field` e valor default ao mesmo tempo (o default vai dentro do próprio `Field`).

Vejamos como usar `Field` para definir **aliases** (apelidos) para os campos.

In [1]:
from pydantic import BaseModel, Field, ValidationError

In [2]:
class Model(BaseModel):
    id_: int = Field(alias="id")
    last_name: str = Field(alias="lastName")

Agora podemos desserializar dados que usam os aliases, mapeando-os para nomes de campo mais "pythônicos".

In [3]:
json_data = """
{
    "id": 100,
    "lastName": "Gauss"
}
"""

m = Model.model_validate_json(json_data)

In [4]:
print(f"m: {m!r}")

m: Model(id_=100, last_name='Gauss')


Os campos foram preenchidos corretamente.

Ou seja, os dados desserializados **precisam** usar os aliases, não os nomes dos campos — inclusive ao criar a instância com argumentos
nomeados, não só ao desserializar um dicionário ou JSON.

In [5]:
print(f'Model(id=100, lastName="Gauss"): {Model(id=100, lastName="Gauss")!r}')

Model(id=100, lastName="Gauss"): Model(id_=100, last_name='Gauss')


In [6]:
try:
    Model(id_=100, last_name="Gauss")
except ValidationError as ex:
    print(f"{type(ex).__name__}: {ex}")

ValidationError: 2 validation errors for Model
id
  Field required [type=missing, input_value={'id_': 100, 'last_name': 'Gauss'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
lastName
  Field required [type=missing, input_value={'id_': 100, 'last_name': 'Gauss'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


E para acessar os campos na instância: alias ou nome do campo?

O modelo é uma classe Python comum, e os atributos são os **nomes dos campos**, não os aliases.

Então usamos os nomes dos campos:

In [7]:
print(f"m.last_name: {m.last_name!r}")

m.last_name: 'Gauss'


E, de fato, `lastName` **não** é atributo da instância:

In [8]:
print(f'hasattr(m, "lastName"): {hasattr(m, "lastName")!r}')

hasattr(m, "lastName"): False


In [9]:
print(f"dict(m): {dict(m)!r}")

dict(m): {'id_': 100, 'last_name': 'Gauss'}


E se quisermos também um default para esses campos? Não dá mais para usar a forma anterior; colocamos o default no próprio `Field`:

In [10]:
class Model(BaseModel):
    id_: int = Field(alias="id", default=100)
    last_name: str = Field(alias="lastName")

In [11]:
print(f'Model(lastName="Gauss"): {Model(lastName="Gauss")!r}')

Model(lastName="Gauss"): Model(id_=100, last_name='Gauss')


In [12]:
print(f'Model(id=1, lastName="Newton"): {Model(id=1, lastName="Newton")!r}')

Model(id=1, lastName="Newton"): Model(id_=1, last_name='Newton')


Ao serializar um modelo, são usados os **nomes dos campos**, não os aliases:

In [13]:
print(f"m.model_dump(): {m.model_dump()!r}")

m.model_dump(): {'id_': 100, 'last_name': 'Gauss'}


Ou seja: ao definir um alias, o comportamento padrão do Pydantic é usá-lo **somente na desserialização**.

A representação usa os nomes dos campos, os atributos da classe são os nomes dos campos e os dados serializados usam os nomes dos campos.

Veremos depois como configurar o modelo para aceitar, na desserialização, tanto o alias quanto o nome do campo.

Também podemos indicar como queremos serializar: o padrão é pelos nomes dos campos, mas podemos pedir explicitamente que o Pydantic use
os aliases (quando existirem).

Vejamos:

In [14]:
class Person(BaseModel):
    id_: int = Field(alias="id")
    first_name: str | None = Field(alias="firstName", default=None)
    last_name: str = Field(alias="lastName")
    age: int | None = None

Todos os campos, exceto `age`, têm alias.

Criando uma instância:

In [15]:
isaac = Person(id=1, firstName="Isaac", lastName="Newton", age=84)
print(f"isaac: {isaac!r}")

isaac: Person(id_=1, first_name='Isaac', last_name='Newton', age=84)


In [16]:
print(f"isaac.model_dump(): {isaac.model_dump()!r}")

isaac.model_dump(): {'id_': 1, 'first_name': 'Isaac', 'last_name': 'Newton', 'age': 84}


A serialização acima usou os nomes dos campos. Para usar os aliases (ou o nome do campo, quando não há alias), passamos um argumento aos
métodos de dump:

In [17]:
print(f"isaac.model_dump(by_alias=True): {isaac.model_dump(by_alias=True)!r}")

isaac.model_dump(by_alias=True): {'id': 1, 'firstName': 'Isaac', 'lastName': 'Newton', 'age': 84}


In [18]:
print(f"isaac.model_dump_json(by_alias=True): {isaac.model_dump_json(by_alias=True)!r}")

isaac.model_dump_json(by_alias=True): '{"id":1,"firstName":"Isaac","lastName":"Newton","age":84}'


Vejamos as informações dos campos — elas ficam mais complexas conforme configuramos modelos e campos:

In [19]:
print(f"Person.model_fields: {Person.model_fields!r}")

Person.model_fields: {'id_': FieldInfo(annotation=int, required=True, alias='id', alias_priority=2), 'first_name': FieldInfo(annotation=Union[str, NoneType], required=False, default=None, alias='firstName', alias_priority=2), 'last_name': FieldInfo(annotation=str, required=True, alias='lastName', alias_priority=2), 'age': FieldInfo(annotation=Union[int, NoneType], required=False, default=None)}


Repare numa propriedade: `alias_priority=2`.

Tecnicamente o Pydantic oferece três configurações de alias:
- `alias`
- `validation_alias`
- `serialization_alias`

Quando se usa as três, dá para indicar qual tem precedência na desserialização e na serialização. O autor do curso nunca usa esse
recurso e não encontrou bom motivo para usá-lo — não é coberto aqui, por ser confuso e pouco necessário num curso de fundamentos.

> 📝 **Nota de revisão (Pydantic 2.11+):** a configuração `populate_by_name`, que o curso usa adiante, foi complementada por
> `validate_by_name` / `validate_by_alias`, e há `serialize_by_alias` para serializar por alias sem passar `by_alias=True` em cada dump.
> Veja a aula 04 desta seção.